In [1]:
from pathlib import Path

import pandas as pd

In [2]:
INPUT_PATH = Path("../data/df_featured.csv")
OUTPUT_DIR = Path("../data")

NUMERIC_FEATURE_COLS = [
    "elo_diff",
    "winrate_10_diff",
    "winrate_30_diff",
    "experience_diff",
    "rank_diff",
    "h2h_winrate",
]
TARGET_COL = "team_1_wins"
TRAIN_RATIO = 0.8

In [3]:
df = pd.read_csv(INPUT_PATH, index_col=0, parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)
print(f"Loaded {len(df)} rows | date range: {df['date'].min().date()} → {df['date'].max().date()}")

assert df.isnull().sum().sum() == 0, "Unexpected NaN values found"

print(f"Maps ({df['_map'].nunique()}): {sorted(df['_map'].unique())}")
print(f"team_1 / team_2: {df['team_1'].nunique()} / {df['team_2'].nunique()} unique — skipped (high cardinality, covered by Elo features)")

Loaded 45773 rows | date range: 2015-11-03 → 2020-03-18
Maps (10): ['Cache', 'Cobblestone', 'Default', 'Dust2', 'Inferno', 'Mirage', 'Nuke', 'Overpass', 'Train', 'Vertigo']
team_1 / team_2: 1248 / 1408 unique — skipped (high cardinality, covered by Elo features)


In [4]:
# _map encoding is deferred to the sklearn Pipeline in step 03 (OneHotEncoder).
# Keeping _map as a string column here ensures the fitted encoder is bundled
# with the model and handles unseen maps gracefully at inference time.
FEATURE_COLS = NUMERIC_FEATURE_COLS + ["_map"]
print(f"Feature columns ({len(FEATURE_COLS)}): {FEATURE_COLS}")

Feature columns (7): ['elo_diff', 'winrate_10_diff', 'winrate_30_diff', 'experience_diff', 'rank_diff', 'h2h_winrate', '_map']


In [5]:
# Chronological split — no shuffling to avoid leaking future data into training
split_idx = int(len(df) * TRAIN_RATIO)
split_date = df.iloc[split_idx]["date"]

train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET_COL]
X_test = test_df[FEATURE_COLS]
y_test = test_df[TARGET_COL]

print(f"Train: {len(X_train)} rows (up to {train_df['date'].max().date()})")
print(f"Test:  {len(X_test)} rows (from {test_df['date'].min().date()})")
print(f"Split date: {split_date.date()}")
print(f"Class balance — train: {y_train.mean():.3f} | test: {y_test.mean():.3f}")

Train: 36618 rows (up to 2019-05-22)
Test:  9155 rows (from 2019-05-22)
Split date: 2019-05-22
Class balance — train: 0.536 | test: 0.536


In [6]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

X_train.to_csv(OUTPUT_DIR / "X_train.csv")
X_test.to_csv(OUTPUT_DIR / "X_test.csv")
y_train.to_csv(OUTPUT_DIR / "y_train.csv")
y_test.to_csv(OUTPUT_DIR / "y_test.csv")

print("Saved:")
for name in ["X_train.csv", "X_test.csv", "y_train.csv", "y_test.csv"]:
    path = OUTPUT_DIR / name
    print(f"  {path}  ({path.stat().st_size // 1024} KB)")

Saved:
  ../data/X_train.csv  (2572 KB)
  ../data/X_test.csv  (619 KB)
  ../data/y_train.csv  (275 KB)
  ../data/y_test.csv  (71 KB)
